In [1]:
from pathlib import Path

import matplotlib.pyplot as pl
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split

import shap

c:\Users\trini\OneDrive\Documents\SC4052\SC4052 Project2\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path= r"C:\Users\trini\.cache\kagglehub\datasets\paololol\league-of-legends-ranked-matches\versions\9"

path = Path(path)
matches = pd.read_csv(path/"matches.csv")
participants = pd.read_csv(path / "participants.csv")
stats1 = pd.read_csv(path / "stats1.csv", low_memory=False)
stats2 = pd.read_csv(path / "stats2.csv", low_memory=False)
stats = pd.concat([stats1, stats2])

In [3]:
a = pd.merge(participants, matches, left_on="matchid", right_on="id", suffixes=("", "_matches"))

print(f"a shape: {a.shape}")
print(f"stats shape: {stats.shape}")

# check for duplicates
print(f"duplicate matchids in a: {a['matchid'].duplicated().sum()}")
print(f"duplicate ids in stats: {stats['id'].duplicated().sum()}")
allstats_orig = pd.merge(a, stats, left_on="matchid", right_on="id", suffixes=("", "_stats"))

a shape: (1834520, 16)
stats shape: (1834517, 56)
duplicate matchids in a: 1650451
duplicate ids in stats: 0


In [4]:
allstats_orig.columns

Index(['id', 'matchid', 'player', 'championid', 'ss1', 'ss2', 'role',
       'position', 'id_matches', 'gameid', 'platformid', 'queueid', 'seasonid',
       'duration', 'creation', 'version', 'id_stats', 'win', 'item1', 'item2',
       'item3', 'item4', 'item5', 'item6', 'trinket', 'kills', 'deaths',
       'assists', 'largestkillingspree', 'largestmultikill', 'killingsprees',
       'longesttimespentliving', 'doublekills', 'triplekills', 'quadrakills',
       'pentakills', 'legendarykills', 'totdmgdealt', 'magicdmgdealt',
       'physicaldmgdealt', 'truedmgdealt', 'largestcrit', 'totdmgtochamp',
       'magicdmgtochamp', 'physdmgtochamp', 'truedmgtochamp', 'totheal',
       'totunitshealed', 'dmgselfmit', 'dmgtoobj', 'dmgtoturrets',
       'visionscore', 'timecc', 'totdmgtaken', 'magicdmgtaken', 'physdmgtaken',
       'truedmgtaken', 'goldearned', 'goldspent', 'turretkills', 'inhibkills',
       'totminionskilled', 'neutralminionskilled', 'ownjunglekills',
       'enemyjunglekills', '

In [10]:
allstats = allstats_orig.copy()

In [11]:
# drop games that lasted less than 10 minutes
allstats = allstats.loc[allstats["duration"] >= 10 * 60, :]

In [12]:
COLS_TO_DROP = [
    # items/spells
    'item1', 'item2', 'item3', 'item4', 'item5', 'item6', 'trinket',
    'ss1', 'ss2',

    # metadata
    'championid', 'queueid', 'seasonid', 'id', 'matchid', 'player', 
    'id_matches', 'gameid', 'id_stats', 'creation', 'version', 'platformid',

    # breakdown damage types — totdmgtochamp already covers this
    'magicdmgtochamp', 'physdmgtochamp', 'truedmgtochamp',
    'magicdmgdealt', 'physicaldmgdealt', 'truedmgdealt', 'totdmgdealt',
    'magicdmgtaken', 'physdmgtaken', 'truedmgtaken',

    # multi-kill counts — kills already captures this
    'largestkillingspree', 'killingsprees', 'largestmultikill',
    'doublekills', 'triplekills', 'quadrakills', 'pentakills', 'legendarykills',

    # not meaningful for win prediction
    'largestcrit', 'longesttimespentliving', 'totunitshealed',
]

In [13]:
allstats.drop(columns=COLS_TO_DROP, inplace=True)

In [14]:
allstats.columns

Index(['role', 'position', 'duration', 'win', 'kills', 'deaths', 'assists',
       'totdmgtochamp', 'totheal', 'dmgselfmit', 'dmgtoobj', 'dmgtoturrets',
       'visionscore', 'timecc', 'totdmgtaken', 'goldearned', 'goldspent',
       'turretkills', 'inhibkills', 'totminionskilled', 'neutralminionskilled',
       'ownjunglekills', 'enemyjunglekills', 'totcctimedealt', 'champlvl',
       'pinksbought', 'wardsbought', 'wardsplaced', 'wardskilled',
       'firstblood'],
      dtype='object')

In [15]:
# adding cs_per_min
allstats["cs_per_min"] = (allstats["totminionskilled"] + allstats["neutralminionskilled"]) / (allstats["duration"] / 60)

In [16]:
# drop the raw cs columns since cs_per_min is enough
allstats.drop(columns=['totminionskilled', 'neutralminionskilled'], inplace=True)
allstats.shape

(1537132, 29)

In [17]:
FEATURES = [c for c in allstats.columns if c != 'win']
FEATURES

['role',
 'position',
 'duration',
 'kills',
 'deaths',
 'assists',
 'totdmgtochamp',
 'totheal',
 'dmgselfmit',
 'dmgtoobj',
 'dmgtoturrets',
 'visionscore',
 'timecc',
 'totdmgtaken',
 'goldearned',
 'goldspent',
 'turretkills',
 'inhibkills',
 'ownjunglekills',
 'enemyjunglekills',
 'totcctimedealt',
 'champlvl',
 'pinksbought',
 'wardsbought',
 'wardsplaced',
 'wardskilled',
 'firstblood',
 'cs_per_min']

In [18]:
cat_cols = allstats.select_dtypes(include='object').columns.tolist()
cat_cols

['role', 'position', 'wardsbought']

In [19]:
# wardsbought should be converted to numeric
allstats["wardsbought"] = allstats["wardsbought"].astype(np.int32)

In [20]:
# role: SOLO / NONE (for jungle) / DUO_CARRY / DUO_SUPPORT
# position: BOT / JUNGLE / TOP / MID

# change position to TOP / JUNGLE / MID / BOT / SUPPORT
allstats['position'] = allstats.apply(
    lambda row: 'SUPPORT' if row['role'] == 'DUO_SUPPORT' else row['position'], 
    axis=1
)

# verify
print(allstats['position'].value_counts())

position
BOT        321759
JUNGLE     314660
MID        305183
TOP        303952
SUPPORT    291578
Name: count, dtype: int64


In [21]:
allstats.drop(columns=['role'], inplace=True)

In [22]:
allstats

,position,duration,win,kills,deaths,assists,totdmgtochamp,totheal,dmgselfmit,dmgtoobj,...,ownjunglekills,enemyjunglekills,totcctimedealt,champlvl,pinksbought,wardsbought,wardsplaced,wardskilled,firstblood,cs_per_min
0,JUNGLE,1909,0,0,2,12,8478,11707,9402,1943,...,1,0,211,14,1,0,17,3,0,0.565741
1,SUPPORT,1909,0,0,2,12,8478,11707,9402,1943,...,1,0,211,14,1,0,17,3,0,0.565741
2,BOT,1909,0,0,2,12,8478,11707,9402,1943,...,1,0,211,14,1,0,17,3,0,0.565741
3,TOP,1909,0,0,2,12,8478,11707,9402,1943,...,1,0,211,14,1,0,17,3,0,0.565741
4,MID,1909,0,0,2,12,8478,11707,9402,1943,...,1,0,211,14,1,0,17,3,0,0.565741
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1579405,BOT,2534,0,2,7,9,11201,6419,0,0,...,1,2,1473,15,0,0,17,6,0,2.107340
1579406,MID,2534,0,2,7,9,11201,6419,0,0,...,1,2,1473,15,0,0,17,6,0,2.107340
1579407,SUPPORT,2534,0,2,7,9,11201,6419,0,0,...,1,2,1473,15,0,0,17,6,0,2.107340
1579408,JUNGLE,2534,0,2,7,9,11201,6419,0,0,...,1,2,1473,15,0,0,17,6,0,2.107340


### Training by position

In [32]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import os

In [26]:
stats_top = allstats[allstats.position == 'TOP']
stats_jungle = allstats[allstats.position == 'JUNGLE'] 
stats_mid = allstats[allstats.position == 'MID']
stats_bot = allstats[allstats.position == 'BOT']
stats_support = allstats[allstats.position == 'SUPPORT']

In [30]:
# Training for TOP 
X_top = stats_top.drop(columns=["win", "position"])
y_top = stats_top["win"]

Xt_top, Xv_top, yt_top, yv_top = train_test_split(X_top, y_top, test_size=0.2, random_state=10)
dt_top = xgb.DMatrix(Xt_top, label=yt_top.values)
dv_top = xgb.DMatrix(Xv_top, label=yv_top.values)

params = {
    "objective": "binary:logistic",
    "base_score": np.mean(yt_top),
    "eval_metric": "logloss",
}
model = xgb.train(
    params,
    dt_top,
    num_boost_round=10,
    evals=[(dt_top, "train"), (dv_top, "valid")],
    early_stopping_rounds=5,
    verbose_eval=25,
)

# get predictions on validation set
y_pred_prob_top = model.predict(dv_top)
y_pred_top = (y_pred_prob_top >= 0.5).astype(int)

# accuracy
accuracy_top = accuracy_score(yv_top, y_pred_top)
print(f"Accuracy: {accuracy_top:.2%}")

[0]	train-logloss:0.57027	valid-logloss:0.57086
[9]	train-logloss:0.34209	valid-logloss:0.34563
Accuracy: 84.67%


In [33]:
os.makedirs('../ml/models', exist_ok=True)

joblib.dump(model, '../ml/models/model_top.pkl')
print("Model saved!")

Model saved!


In [34]:
# Training for JUNGLE
X_jungle = stats_jungle.drop(columns=["win", "position"])
y_jungle = stats_jungle["win"]

Xt_jungle, Xv_jungle, yt_jungle, yv_jungle = train_test_split(X_jungle, y_jungle, test_size=0.2, random_state=10)
dt_jungle = xgb.DMatrix(Xt_jungle, label=yt_jungle.values)
dv_jungle = xgb.DMatrix(Xv_jungle, label=yv_jungle.values)

params = {
    "objective": "binary:logistic",
    "base_score": np.mean(yt_jungle),
    "eval_metric": "logloss",
}
model = xgb.train(
    params,
    dt_jungle,
    num_boost_round=10,
    evals=[(dt_jungle, "train"), (dv_jungle, "valid")],
    early_stopping_rounds=5,
    verbose_eval=25,
)

# get predictions on validation set
y_pred_prob_jungle = model.predict(dv_jungle)
y_pred_jungle = (y_pred_prob_jungle >= 0.5).astype(int)

# accuracy
accuracy_jungle = accuracy_score(yv_jungle, y_pred_jungle)
print(f"Accuracy: {accuracy_jungle:.2%}")

[0]	train-logloss:0.56980	valid-logloss:0.57028
[9]	train-logloss:0.34231	valid-logloss:0.34656
Accuracy: 84.48%


In [35]:
os.makedirs('../ml/models', exist_ok=True)

joblib.dump(model, '../ml/models/model_jungle.pkl')
print("Model saved!")

Model saved!


In [37]:
# Training for MID
X_mid = stats_mid.drop(columns=["win", "position"])
y_mid = stats_mid["win"]

Xt_mid, Xv_mid, yt_mid, yv_mid = train_test_split(X_mid, y_mid, test_size=0.2, random_state=10)
dt_mid = xgb.DMatrix(Xt_mid, label=yt_mid.values)
dv_mid = xgb.DMatrix(Xv_mid, label=yv_mid.values)

params = {
    "objective": "binary:logistic",
    "base_score": np.mean(yt_mid),
    "eval_metric": "logloss",
}
model = xgb.train(
    params,
    dt_mid,
    num_boost_round=10,
    evals=[(dt_mid, "train"), (dv_mid, "valid")],
    early_stopping_rounds=5,
    verbose_eval=25,
)

# get predictions on validation set
y_pred_prob_mid = model.predict(dv_mid)
y_pred_mid = (y_pred_prob_mid >= 0.5).astype(int)

# accuracy
accuracy_mid = accuracy_score(yv_mid, y_pred_mid)
print(f"Accuracy: {accuracy_mid:.2%}")

[0]	train-logloss:0.57006	valid-logloss:0.57004
[9]	train-logloss:0.34434	valid-logloss:0.34775
Accuracy: 84.55%


In [38]:
os.makedirs('../ml/models', exist_ok=True)

joblib.dump(model, '../ml/models/model_mid.pkl')
print("Model saved!")

Model saved!


In [39]:
# Training for BOT
X_bot = stats_bot.drop(columns=["win", "position"])
y_bot = stats_bot["win"]

Xt_bot, Xv_bot, yt_bot, yv_bot = train_test_split(X_bot, y_bot, test_size=0.2, random_state=10)
dt_bot = xgb.DMatrix(Xt_bot, label=yt_bot.values)
dv_bot = xgb.DMatrix(Xv_bot, label=yv_bot.values)

params = {
    "objective": "binary:logistic",
    "base_score": np.mean(yt_bot),
    "eval_metric": "logloss",
}
model = xgb.train(
    params,
    dt_bot,
    num_boost_round=10,
    evals=[(dt_bot, "train"), (dv_bot, "valid")],
    early_stopping_rounds=5,
    verbose_eval=25,
)

# get predictions on validation set
y_pred_prob_bot = model.predict(dv_bot)
y_pred_bot = (y_pred_prob_bot >= 0.5).astype(int)

# accuracy
accuracy_bot = accuracy_score(yv_bot, y_pred_bot)
print(f"Accuracy: {accuracy_bot:.2%}")

[0]	train-logloss:0.56953	valid-logloss:0.57115
[9]	train-logloss:0.34155	valid-logloss:0.34808
Accuracy: 84.57%


In [40]:
os.makedirs('../ml/models', exist_ok=True)

joblib.dump(model, '../ml/models/model_bot.pkl')
print("Model saved!")

Model saved!


In [41]:
# Training for SUPPORT
X_support = stats_support.drop(columns=["win", "position"])
y_support = stats_support["win"]

Xt_support, Xv_support, yt_support, yv_support = train_test_split(X_support, y_support, test_size=0.2, random_state=10)
dt_support = xgb.DMatrix(Xt_support, label=yt_support.values)
dv_support = xgb.DMatrix(Xv_support, label=yv_support.values)

params = {
    "objective": "binary:logistic",
    "base_score": np.mean(yt_support),
    "eval_metric": "logloss",
}
model = xgb.train(
    params,
    dt_support,
    num_boost_round=10,
    evals=[(dt_support, "train"), (dv_support, "valid")],
    early_stopping_rounds=5,
    verbose_eval=25,
)

# get predictions on validation set
y_pred_prob_support = model.predict(dv_support)
y_pred_support = (y_pred_prob_support >= 0.5).astype(int)

# accuracy
accuracy_support = accuracy_score(yv_support, y_pred_support)
print(f"Accuracy: {accuracy_support:.2%}")

[0]	train-logloss:0.56990	valid-logloss:0.57069
[9]	train-logloss:0.34287	valid-logloss:0.34802
Accuracy: 84.52%


In [42]:
os.makedirs('../ml/models', exist_ok=True)

joblib.dump(model, '../ml/models/model_support.pkl')
print("Model saved!")

Model saved!


In [ ]:
# columns used for training
stats_top.columns

Index(['position', 'duration', 'win', 'kills', 'deaths', 'assists',
       'totdmgtochamp', 'totheal', 'dmgselfmit', 'dmgtoobj', 'dmgtoturrets',
       'visionscore', 'timecc', 'totdmgtaken', 'goldearned', 'goldspent',
       'turretkills', 'inhibkills', 'ownjunglekills', 'enemyjunglekills',
       'totcctimedealt', 'champlvl', 'pinksbought', 'wardsbought',
       'wardsplaced', 'wardskilled', 'firstblood', 'cs_per_min'],
      dtype='object')

In [47]:
len(stats_top.columns)

28